# 3.nivelSesionReprAcustica · 1. Creación de espectrogramas (representación para CNN)

Vía **"caja negra" tipo estado del arte**: en lugar de features acústicas interpretables, se
calcula una **representación espectral** (log-mel-espectrograma) por ventana de habla del
paciente, para alimentar una CNN (siguiente notebook). Sirve como **comparativa de rendimiento**
frente al Random Forest interpretable — el eje rendimiento↔interpretabilidad del trabajo.

**Anclaje en la literatura del propio DAIC-WOZ** (búsqueda 2026):
- Segmentos de **4 s** a **16 kHz**; espectrograma logarítmico normalizado a [0,1]; predicción
  por segmento **agregada a sujeto** promediando probabilidades (Vázquez-Romero & Gallardo-Antolín,
  *Ensemble 1D-CNN*, Entropy 2020; DepAudioNet, Ma et al. 2016).
- Resolución típica de imagen espectral **128×128** (128 bandas mel × 128 frames).
- Rendimiento honesto SOTA audio-only: **F1 clase deprimida ≈ 0.5–0.65, AUC/accuracy ≈ 0.74**
  a nivel de sujeto (Ensemble 1D-CNN: acc 0.74, F1 0.65). Los ~0.90+ que se ven publicados son
  **a nivel de segmento** o con fugas de validez (prompts del terapeuta) — a nivel de sujeto y
  sin fuga, el rango honesto es el anterior.

**Decisiones de diseño (para comparabilidad con el resto del TFE):**
- Se reutiliza el **mismo aislamiento de habla del paciente** (timestamps del transcript) y el
  **mismo ventaneo de 4 s** que en `2.nivelSegmento`, de modo que la CNN opera sobre las MISMAS
  unidades que el RF de segmento → comparación directa tras agregar a sesión.
- **NO** se normaliza por sexo ni se seleccionan variables: la CNN recibe el espectrograma crudo
  (normalizado [0,1]). Es, por diseño, la aproximación no interpretable.

**Formato de salida:** los espectrogramas son arrays 2D (128×128), así que NO caben en un CSV
de escalares. Se guardan como un único `.npy` apilado `(N, 128, 128)` + un **CSV manifiesto**
(una fila por ventana, con participant_id/split/phq8/…) que los enlaza. El manifiesto es el
equivalente al CSV de features de segmento.

## 0. Setup y parámetros

In [1]:
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

# --- rutas ---
from pathlib import Path
PROJECT_DIR = Path.home().as_posix() + '/Desktop/Master/UNIR_IA_TFE'   # ruta local del proyecto
path_output = PROJECT_DIR + '/output'
SPEC_DIR = path_output + '/spectrograms'
os.makedirs(SPEC_DIR, exist_ok=True)

# --- parámetros de aislamiento/ventaneo (idénticos a 2.nivelSegmento) ---
SR          = 16_000
FRAME_LEN   = int(0.025 * SR)   # usado por load_patient_segments para descartar segmentos ínfimos
MIN_SEG_DUR = 0.3               # s — segmento mínimo del paciente
WIN_DUR = 4.0                   # s — ventana
WIN_HOP = 4.0                   # s — sin solape
MIN_WIN = 2.0                   # s — se conserva el resto final si dura ≥ esto

# --- parámetros del log-mel-espectrograma (128×128, anclados en literatura) ---
N_MELS     = 128
N_FFT      = 1024               # 64 ms a 16 kHz
HOP_LENGTH = 500                # 4 s / 500 ≈ 128 frames
N_FRAMES   = 128                # eje temporal fijo (pad/truncado)
F_MIN, F_MAX = 20, 8000

In [2]:
df_sample = pd.read_csv(path_output + '/df_sample.csv')
print(f"Sesiones: {len(df_sample)}")

Sesiones: 186


## 1. Aislamiento de habla del paciente (reutilizado verbatim) y ventaneo

In [3]:
def load_patient_segments(path_audio: str,
                           path_transcript: str,
                           speaker: str = 'Participant') -> list:
    """
    Carga el audio y extrae únicamente los segmentos de habla verbal del paciente.

    Decisión: aislar el habla del paciente via timestamps del transcript (ground
    truth anotado), en lugar de VAD automático. Esto evita contaminar las
    features con la voz de Ellie o los silencios interturno, que son
    acústicamente distintos a la habla del paciente y distorsionarían
    métricas prosódicas y de calidad vocal.

    Se excluyen marcadores no verbales (<laughter>, <cough>, <synch>, etc.)
    para no contaminar features acústicas con audio que no es habla.

    Se descartan segmentos < MIN_SEG_DUR (0.3 s) para evitar artefactos en
    el cálculo de MFCCs y estimación de F0.
    """
    df_t = pd.read_csv(path_transcript, sep='\t')
    df_t.columns = df_t.columns.str.strip()
    df_t = (df_t[df_t['speaker'] == speaker]
              .sort_values('start_time')
              .reset_index(drop=True))

    # Excluir marcadores no verbales: <laughter>, <cough>, <synch>, etc.
    df_t = df_t[~df_t['value'].str.strip().str.startswith('<', na=False)].reset_index(drop=True)

    y, _ = librosa.load(path_audio, sr=SR, mono=True)

    segments = []
    for _, row in df_t.iterrows():
        dur = row['stop_time'] - row['start_time']
        if dur < MIN_SEG_DUR:
            continue

        s = int(row['start_time'] * SR)
        e = int(row['stop_time']  * SR)
        seg = y[s:e]

        if len(seg) >= FRAME_LEN:
            segments.append(seg)

    return segments


## ─────────────────────────────────────────────────────────────────────────────

In [4]:
def to_windows(segments):
    """Corta cada segmento en ventanas de WIN_DUR s sin solape; conserva el resto
    final si dura >= MIN_WIN s. Ventanea DENTRO de cada segmento (mismo criterio que 2.nivelSegmento)."""
    win, hop, minw = int(WIN_DUR * SR), int(WIN_HOP * SR), int(MIN_WIN * SR)
    out = []
    for seg in segments:
        n = len(seg)
        if n < minw:
            continue
        start = 0
        while start < n:
            w = seg[start:start + win]
            if len(w) >= minw:
                out.append(w)
            start += hop
    return out

## 2. Log-mel-espectrograma por ventana (128×128)

In [5]:
def window_to_logmel(w):
    """Ventana de audio -> log-mel-espectrograma 128×128 normalizado a [0,1]."""
    S = librosa.feature.melspectrogram(
        y=w.astype(np.float32), sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX, power=2.0)
    S_db = librosa.power_to_db(S, ref=np.max)          # escala logarítmica (dB)
    # fijar el eje temporal a N_FRAMES (pad con el mínimo = silencio, o truncar)
    T = S_db.shape[1]
    if T < N_FRAMES:
        pad = np.full((N_MELS, N_FRAMES - T), S_db.min(), dtype=S_db.dtype)
        S_db = np.concatenate([S_db, pad], axis=1)
    else:
        S_db = S_db[:, :N_FRAMES]
    # normalización min-max a [0,1] por espectrograma (como en la literatura)
    mn, mx = S_db.min(), S_db.max()
    return ((S_db - mn) / (mx - mn + 1e-8)).astype(np.float32)   # (128, 128)

# sanity check en una ventana
_seg = load_patient_segments(df_sample.iloc[0]['path_audio'], df_sample.iloc[0]['path_transcript'])
_w = to_windows(_seg)[0]
_S = window_to_logmel(_w)
print("Shape de un espectrograma:", _S.shape, "| rango:", round(float(_S.min()),2), "-", round(float(_S.max()),2))

Shape de un espectrograma: (128, 128) | rango: 0.0 - 1.0


## 3. Extracción sobre toda la muestra

In [6]:
specs, manifest = [], []
META = ['participant_id', 'phq8_score', 'phq8_binary', 'gender', 'split']

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Sesiones"):
    try:
        segments = load_patient_segments(row['path_audio'], row['path_transcript'])
    except Exception as e:
        print(f"  [{row.get('participant_id','?')}] ERROR: {e}")
        continue
    for wi, w in enumerate(to_windows(segments)):
        specs.append(window_to_logmel(w))
        rec = {c: row[c] for c in META}
        rec['window_idx'] = wi
        manifest.append(rec)

X = np.stack(specs).astype(np.float32)          # (N, 128, 128)
df_manifest = pd.DataFrame(manifest)
# el orden de X y df_manifest coincide fila a fila (row_id implícito = posición)
df_manifest.insert(0, 'row_id', np.arange(len(df_manifest)))

print(f"\nEspectrogramas: {X.shape}  (~{X.nbytes/1e9:.2f} GB en float32)")
print(f"Ventanas por sesión: media {df_manifest.groupby('participant_id').size().mean():.1f}")
print(f"Sesiones cubiertas: {df_manifest['participant_id'].nunique()} / {len(df_sample)}")

Sesiones: 100%|██████████| 186/186 [01:18<00:00,  2.36it/s]



Espectrogramas: (19132, 128, 128)  (~1.25 GB en float32)
Ventanas por sesión: media 102.9
Sesiones cubiertas: 186 / 186


## 4. Guardado (npy apilado + CSV manifiesto)

In [7]:
np.save(path_output + '/spectrograms_128.npy', X)
df_manifest.to_csv(path_output + '/df_spectrogram_manifest.csv', index=False)
print("Guardado:")
print(f"  {path_output}/spectrograms_128.npy         shape {X.shape}")
print(f"  {path_output}/df_spectrogram_manifest.csv   filas {len(df_manifest)}")
print("\nEn el notebook de la CNN: X = np.load('.../spectrograms_128.npy'); "
      "manifest = pd.read_csv('.../df_spectrogram_manifest.csv')  # alineados por row_id")

Guardado:
  C:/Users/tblxa/Desktop/Master/UNIR_IA_TFE/output/spectrograms_128.npy         shape (19132, 128, 128)
  C:/Users/tblxa/Desktop/Master/UNIR_IA_TFE/output/df_spectrogram_manifest.csv   filas 19132

En el notebook de la CNN: X = np.load('.../spectrograms_128.npy'); manifest = pd.read_csv('.../df_spectrogram_manifest.csv')  # alineados por row_id
